In [1]:
import pandas as pd
df_golden = pd.read_csv("kalshi_weather_matrix.csv")

In [2]:
pd.set_option('display.max_columns', None)
df_golden.head()

,ticker,local_date,hour,hourly_avg_yes_price,hourly_avg_no_price,hourly_volume,time,temp,rhum,wspd,pres,cldc,wdir_sin,wdir_cos,condition_1,condition_2,condition_3,condition_4,condition_5,condition_7,condition_8,condition_9,condition_18,condition_25,target_win
0,KXHIGHLAX-25APR01-B60.5,2025-04-01,4,16.680000,83.320000,1162,2025-04-01 04:00:00-08:00,53.96,83.0,18.4,1014.9,4.0,-1.000000,1.192488e-08,0,1,0,0,0,0,0,0,0,0,0.0
1,KXHIGHLAX-25APR01-B60.5,2025-04-01,5,15.733333,84.266667,190,2025-04-01 05:00:00-08:00,53.96,72.0,7.6,1015.0,4.0,-0.766045,6.427875e-01,0,0,1,0,0,0,0,0,0,0,0.0
2,KXHIGHLAX-25APR01-B60.5,2025-04-01,6,13.333333,86.666667,140,2025-04-01 06:00:00-08:00,53.06,74.0,11.2,1015.3,6.0,-0.866025,4.999999e-01,0,0,1,0,0,0,0,0,0,0,0.0
3,KXHIGHLAX-25APR01-B60.5,2025-04-01,7,10.900000,89.100000,47,2025-04-01 07:00:00-08:00,55.04,72.0,16.6,1015.8,6.0,-0.984808,1.736482e-01,0,0,1,0,0,0,0,0,0,0,0.0
4,KXHIGHLAX-25APR01-B60.5,2025-04-01,8,7.222222,92.777778,292,2025-04-01 08:00:00-08:00,57.02,64.0,18.4,1015.9,4.0,-0.984808,1.736482e-01,0,0,1,0,0,0,0,0,0,0,0.0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Prepping data for Machine Learning...")

# 1. Sort chronologically just to be absolutely safe
df_golden = df_golden.sort_values(by=['local_date', 'hour'])

df_golden = df_golden.dropna(subset=['target_win'])
df_golden['target_win'] = df_golden['target_win'].astype(int)
# 2. Define your Features (X) and Target (y)
# We drop the target AND the string/date metadata columns from X
X = df_golden.drop(columns=['target_win', 'ticker', 'local_date', 'time'])
y = df_golden['target_win']

# 3. Chronological Train/Test Split (NO SHUFFLING!)
# We use the first 80% of the year to train, and the last 20% to test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=False)

print(f"Training on {len(X_train)} rows, Testing on {len(X_test)} rows.\n")

# 4. Initialize and Train the Baseline Model
print("Training Random Forest Baseline...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 5. Make Predictions on the unseen Test data
y_pred = rf_model.predict(X_test)

# 6. Evaluate!
print("\n--- Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Prepping data for Machine Learning...
Training on 25805 rows, Testing on 6452 rows.

Training Random Forest Baseline...

--- Model Performance ---
Accuracy: 90.89%

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.97      0.95      5269
           1       0.83      0.63      0.72      1183

    accuracy                           0.91      6452
   macro avg       0.88      0.80      0.83      6452
weighted avg       0.90      0.91      0.90      6452



In [4]:
# THE FIX: Drop the pricing data so the model is forced to learn the weather!
X = df_golden.drop(columns=[
    'target_win', 
    'ticker', 
    'local_date', 
    'time', 
    'hourly_avg_yes_price', 
    'hourly_avg_no_price', 
    'hourly_volume'
])
y = df_golden['target_win']

# Re-run the Train/Test Split and Model Fitting with this new X
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=False)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("\n--- PURE WEATHER Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))


--- PURE WEATHER Model Performance ---
Accuracy: 80.27%
              precision    recall  f1-score   support

           0       0.82      0.98      0.89      5269
           1       0.20      0.03      0.05      1183

    accuracy                           0.80      6452
   macro avg       0.51      0.50      0.47      6452
weighted avg       0.70      0.80      0.74      6452



In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Applying Class Weights to force the model to find winners...")

# 1. Define your Features (X) and Target (y)
X = df_golden.drop(columns=[
    'target_win', 
    'ticker', 
    'local_date', 
    'time', 
    'hourly_avg_yes_price', 
    'hourly_avg_no_price', 
    'hourly_volume'
])
y = df_golden['target_win']

# 2. Chronological Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=False)

# 3. THE FIX: Add class_weight='balanced' to the model
print("Training Balanced Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    n_jobs=-1, 
    class_weight='balanced' # <--- THIS IS THE MAGIC WAND
)
rf_model.fit(X_train, y_train)

# 4. Make Predictions and Evaluate
y_pred = rf_model.predict(X_test)

print("\n--- BALANCED Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

Applying Class Weights to force the model to find winners...
Training Balanced Random Forest...

--- BALANCED Model Performance ---
Accuracy: 76.46%
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      5269
           1       0.21      0.10      0.13      1183

    accuracy                           0.76      6452
   macro avg       0.51      0.51      0.50      6452
weighted avg       0.71      0.76      0.73      6452



In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Isolating the Morning Trading Window (6 AM to 11 AM)...")

# 1. Filter the Golden Matrix to ONLY include morning hours
df_morning = df_golden[(df_golden['hour'] >= 6) & (df_golden['hour'] <= 11)].copy()

# 2. Define Features and Target (Still keeping prices out to prevent leakage!)
X_morning = df_morning.drop(columns=[
    'target_win', 
    'ticker', 
    'local_date', 
    'time', 
    'hourly_avg_yes_price', 
    'hourly_avg_no_price', 
    'hourly_volume'
])
y_morning = df_morning['target_win']

# 3. Chronological Split on the new morning-only data
X_train, X_test, y_train, y_test = train_test_split(X_morning, y_morning, test_size=0.20, shuffle=False)

# 4. Train the Balanced Model
print("Training Morning-Only Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
rf_model.fit(X_train, y_train)

# 5. Evaluate
y_pred = rf_model.predict(X_test)

print("\n--- MORNING-ONLY Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

Isolating the Morning Trading Window (6 AM to 11 AM)...
Training Morning-Only Random Forest...

--- MORNING-ONLY Model Performance ---
Accuracy: 81.95%
              precision    recall  f1-score   support

           0       0.82      0.99      0.90      1973
           1       0.17      0.01      0.01       420

    accuracy                           0.82      2393
   macro avg       0.50      0.50      0.46      2393
weighted avg       0.71      0.82      0.74      2393



In [7]:
df_weather = pd.read_csv("weather_only.csv")

In [8]:
import pandas as pd

print("Engineering momentum and historical features...")

# 1. Ensure the weather data is perfectly chronological
df_weather = df_weather.sort_values(by=['local_date', 'hour']).reset_index(drop=True)

# 2. The Heating Curve: Current temp minus the temp 3 hours ago
df_weather['temp_change_3h'] = df_weather['temp'] - df_weather['temp'].shift(3)

# 3. The Morning Peak: Max temp over the last 3 hours
df_weather['rolling_max_3h'] = df_weather['temp'].rolling(window=3).max()

# 4. The Autocorrelation: Yesterday's High
# We calculate the daily highs, shift them down by one day, and merge them back
df_daily_highs = df_weather.groupby('local_date')['temp'].max().reset_index()
df_daily_highs.rename(columns={'temp': 'actual_daily_high'}, inplace=True)
df_daily_highs['yesterday_high'] = df_daily_highs['actual_daily_high'].shift(1)

# Merge 'yesterday_high' back onto the hourly weather data
df_weather = pd.merge(
    df_weather, 
    df_daily_highs[['local_date', 'yesterday_high']], 
    on='local_date', 
    how='left'
)

# 5. Clean up the NaNs
# Because we looked "backwards" 3 hours and 1 day, the very first day in your dataset 
# won't have this historical data. We must drop those initial NaN rows.
df_weather = df_weather.dropna().copy()

print("Feature Engineering Complete! Here are the new columns:")
print(df_weather[['local_date', 'hour', 'temp', 'temp_change_3h', 'yesterday_high']].head(10))

Engineering momentum and historical features...
Feature Engineering Complete! Here are the new columns:
    local_date  hour   temp  temp_change_3h  yesterday_high
8   2025-01-04     0  55.04            1.08            59.0
9   2025-01-04     1  55.94            0.90            59.0
10  2025-01-04     2  51.98           -3.96            59.0
11  2025-01-04     3  51.98           -3.06            59.0
12  2025-01-04     4  51.08           -4.86            59.0
13  2025-01-04     5  50.00           -1.98            59.0
14  2025-01-04     6  50.00           -1.98            59.0
15  2025-01-04     7  51.08            0.00            59.0
16  2025-01-04     8  51.98            1.98            59.0
17  2025-01-04     9  53.96            3.96            59.0


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd

print("Pivoting to Regression: Predicting the exact daily high...")

# 1. Calculate the True Daily High from your clean weather dataset
# We group by the date and find the absolute maximum temp for that day
df_daily_highs = df_weather.groupby('local_date')['temp'].max().reset_index()
df_daily_highs.rename(columns={'temp': 'actual_daily_high'}, inplace=True)

# 2. Merge this new continuous target onto our Morning dataframe
df_morning_reg = pd.merge(
    df_morning, 
    df_daily_highs, 
    on='local_date', 
    how='inner'
)

# 3. Define Features (X) and our new Target (y)
# We drop all metadata, prices, the old classification target, AND the new target
X_reg = df_morning_reg.drop(columns=[
    'target_win', 
    'actual_daily_high', 
    'ticker', 
    'local_date', 
    'time', 
    'hourly_avg_yes_price', 
    'hourly_avg_no_price', 
    'hourly_volume'
])
y_reg = df_morning_reg['actual_daily_high']

# 4. Chronological Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.20, shuffle=False)

# 5. Initialize and Train the REGRESSOR 
print("Training Morning-Only Random Forest Regressor...")
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_regressor.fit(X_train, y_train)

# 6. Make Predictions on the unseen Test data
y_pred = rf_regressor.predict(X_test)

# 7. Evaluate using Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, y_pred)
print("\n--- REGRESSION Model Performance ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}°F")

# Let's print a side-by-side of the first 5 predictions vs reality
comparison_df = pd.DataFrame({'Actual High': y_test, 'Predicted High': y_pred})
print("\nSample Predictions:")
print(comparison_df.head())

Pivoting to Regression: Predicting the exact daily high...
Training Morning-Only Random Forest Regressor...

--- REGRESSION Model Performance ---
Mean Absolute Error (MAE): 3.04°F

Sample Predictions:
      Actual High  Predicted High
9378         62.6         63.5144
9379         62.6         63.5144
9380         62.6         63.5144
9381         62.6         63.5144
9382         62.6         63.5144


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd

print("Building pure-weather Regression dataset...")

# 1. Isolate the Morning Window (e.g., 6 AM to 11 AM) from df_golden
df_morning = df_golden[(df_golden['hour'] >= 6) & (df_golden['hour'] <= 11)].copy()

# 2. Get the true daily high from your raw weather data
df_daily_highs = df_weather.groupby('local_date')['temp'].max().reset_index()
df_daily_highs.rename(columns={'temp': 'actual_daily_high'}, inplace=True)

# 3. Merge the true high onto your morning dataframe
df_morning_reg = pd.merge(df_morning, df_daily_highs, on='local_date', how='inner')

# 4. Define Target (y)
y_reg = df_morning_reg['actual_daily_high']

# 5. Define Features (X) by strictly dropping ALL Kalshi metadata and prices
# Notice we KEEP the 'hour' column. 55°F at 6 AM means something very different than 55°F at 11 AM!
columns_to_drop = [
    'target_win',           # Old classification target
    'actual_daily_high',    # New regression target
    'ticker',               # Kalshi string
    'local_date',           # Date string
    'time',                 # Time string
    'hourly_avg_yes_price', # Leakage: Human pricing
    'hourly_avg_no_price',  # Leakage: Human pricing
    'hourly_volume'         # Leakage: Human volume
]

# Safely drop these columns to leave ONLY weather features + hour
X_reg = df_morning_reg.drop(columns=[col for col in columns_to_drop if col in df_morning_reg.columns])

# 6. Chronological Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.20, shuffle=False)

# 7. Initialize and Train the Regressor
print("Training Pure-Weather Random Forest Regressor...")
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_regressor.fit(X_train, y_train)

# 8. Predict and Evaluate
y_pred = rf_regressor.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print("\n--- REGRESSION Model Performance ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}°F")

# View the side-by-side comparison
comparison_df = pd.DataFrame({'Actual High': y_test.values, 'Predicted High': y_pred})
print("\nSample Predictions:")
print(comparison_df.head())

Building pure-weather Regression dataset...
Training Pure-Weather Random Forest Regressor...

--- REGRESSION Model Performance ---
Mean Absolute Error (MAE): 3.04°F

Sample Predictions:
   Actual High  Predicted High
0         62.6         63.5144
1         62.6         63.5144
2         62.6         63.5144
3         62.6         63.5144
4         62.6         63.5144


In [11]:
import pandas as pd

print("Constructing the Swing Trading Simulator DataFrame...")

# 1. We take our X_test dataframe (which has the weather features)
# and attach our Regressor's predictions to it
simulator_df = X_test.copy()
simulator_df['predicted_high'] = y_pred

# 2. Re-attach the metadata and pricing! 
# We use the index to perfectly match the dropped columns back to our test set
columns_to_reattach = [
    'ticker', 
    'local_date', 
    'hour', 
    'hourly_avg_yes_price', 
    'actual_daily_high' # We keep the truth just to see how we did
]

for col in columns_to_reattach:
    simulator_df[col] = df_morning_reg.loc[simulator_df.index, col]

# 3. Sort chronologically so our bot trades forward through time
simulator_df = simulator_df.sort_values(by=['local_date', 'hour'])

# Let's look at the bot's raw material
print("Simulator Data Ready:")
print(simulator_df[['local_date', 'hour', 'ticker', 'predicted_high', 'hourly_avg_yes_price']].head())

Constructing the Swing Trading Simulator DataFrame...
Simulator Data Ready:
      local_date  hour                   ticker  predicted_high  \
9378  2025-12-16     8  KXHIGHLAX-25DEC16-B73.5         63.5144   
9379  2025-12-16     8  KXHIGHLAX-25DEC16-B75.5         63.5144   
9380  2025-12-16     8  KXHIGHLAX-25DEC16-B77.5         63.5144   
9381  2025-12-16     8    KXHIGHLAX-25DEC16-T71         63.5144   
9382  2025-12-16     8    KXHIGHLAX-25DEC16-T78         63.5144   

      hourly_avg_yes_price  
9378              2.520000  
9379              1.000000  
9380              1.000000  
9381             95.147541  
9382              1.000000  


In [12]:
def check_bracket_match(predicted_temp, ticker):
    """
    Parses Kalshi temperature tickers and checks if the ML prediction 
    falls within the specific contract's boundaries.
    """
    # Extract the final piece of the ticker (e.g., 'B64.5' or 'T73')
    bracket_code = ticker.split('-')[-1]
    
    # 1. Handle the "Between" brackets (e.g., B64.5)
    if bracket_code.startswith('B'):
        # Extract the float number (64.5)
        center_temp = float(bracket_code[1:])
        
        # Calculate the 1-degree bounds (e.g., 64.5 becomes 64.0 to 65.0)
        lower_bound = center_temp - 0.5
        upper_bound = center_temp + 0.5
        
        return lower_bound <= predicted_temp <= upper_bound
        
    # 2. Handle the "Tail" extreme brackets (e.g., T66 or T73)
    elif bracket_code.startswith('T'):
        tail_temp = float(bracket_code[1:])
        
        # If the tail is a high number, it's the "Above X" market.
        # If it's a low number, it's the "Below X" market.
        # (Using 70 as a rough midpoint for Los Angeles weather)
        if tail_temp > 70: 
            return predicted_temp >= tail_temp
        else:
            return predicted_temp <= tail_temp
            
    # Default failsafe
    return False

# Quick test to prove it works!
print(f"Prediction 64.8 in B64.5? {check_bracket_match(64.8, 'KXHIGHLAX-25APR01-B64.5')}")
print(f"Prediction 65.2 in B64.5? {check_bracket_match(65.2, 'KXHIGHLAX-25APR01-B64.5')}")

Prediction 64.8 in B64.5? True
Prediction 65.2 in B64.5? False


In [13]:
print("Initializing Kalshi Swing Trading Simulator...")

starting_bankroll = 100.0
current_bankroll = starting_bankroll
winning_trades = 0
losing_trades = 0

# We group by each specific contract on each specific day
# This isolates the timeline so we can track the price action of one contract through the morning
for (date, ticker), group in simulator_df.groupby(['local_date', 'ticker']):
    
    # Sort the morning hours chronologically (6 AM -> 11 AM)
    market_data = group.sort_values('hour')
    
    buy_price = 0
    holding = False
    
    for index, row in market_data.iterrows():
        current_price = row['hourly_avg_yes_price']
        
        # 1. THE BUY SIGNAL 
        # (In production, you will add a rule here that says: 
        # "Only buy if predicted_high fits inside the text of this specific ticker string")
        # 1. THE BUY SIGNAL 
        # The bot only buys if the price is cheap AND the prediction matches the ticker bounds!
        is_target = check_bracket_match(row['predicted_high'], ticker)
        
        if not holding and is_target and current_price > 15 and current_price < 50: 
            buy_price = current_price
            holding = True
            continue
            
        # 2. THE SELL SIGNALS (Managing the open position)
        if holding:
            # Take-Profit: If price jumps by 15 cents, lock in the win!
            if current_price >= buy_price + 35:
                profit_cents = current_price - buy_price
                current_bankroll += (profit_cents / 100) # Convert cents to dollars
                winning_trades += 1
                holding = False
                break # We made our money. Exit this market for the day.
                
            # Stop-Loss: If it drops by 10 cents, cut our losses.
            elif current_price <= buy_price - 3:
                loss_cents = buy_price - current_price
                current_bankroll -= (loss_cents / 100)
                losing_trades += 1
                holding = False
                break # Exit the market to prevent a total wipeout.
                
    # 3. THE TIME-STOP
    # If we reach 11 AM and are still holding, force a sell to avoid afternoon risk.
    if holding:
        final_price = market_data.iloc[-1]['hourly_avg_yes_price']
        pnl_cents = final_price - buy_price
        current_bankroll += (pnl_cents / 100)
        
        if pnl_cents > 0: 
            winning_trades += 1
        else: 
            losing_trades += 1

print(f"\n--- Swing Trading Simulation Complete ---")
print(f"Total Trades Executed: {winning_trades + losing_trades}")
print(f"Winning Trades: {winning_trades}")
print(f"Losing Trades: {losing_trades}")
print(f"Final Bankroll: ${current_bankroll:.2f}")

Initializing Kalshi Swing Trading Simulator...

--- Swing Trading Simulation Complete ---
Total Trades Executed: 48
Winning Trades: 13
Losing Trades: 35
Final Bankroll: $99.69


In [14]:
import pandas as pd

# 1. The Kalshi Bracket Parser
def check_bracket_match(predicted_temp, ticker):
    """
    Translates Kalshi tickers and mathematically checks if the ML 
    prediction falls inside the exact contract boundaries.
    """
    bracket_code = ticker.split('-')[-1]
    
    try:
        if bracket_code.startswith('B'):
            center_temp = float(bracket_code[1:])
            return (center_temp - 0.5) <= predicted_temp <= (center_temp + 0.5)
            
        elif bracket_code.startswith('T'):
            tail_temp = float(bracket_code[1:])
            # Assuming 70 is a rough midpoint; Kalshi formats High tails differently than Low tails
            if tail_temp > 70: 
                return predicted_temp >= tail_temp
            else:
                return predicted_temp <= tail_temp
    except ValueError:
        return False
        
    return False

# 2. The Trading Exchange Loop
print("Initializing Kalshi Swing Trading Simulator...")

starting_bankroll = 100.0
current_bankroll = starting_bankroll
winning_trades = 0
losing_trades = 0

# Group by date and ticker to isolate individual contract timelines
for (date, ticker), group in simulator_df.groupby(['local_date', 'ticker']):
    
    market_data = group.sort_values('hour')
    
    buy_price = 0
    holding = False
    
    # Step through the morning hour-by-hour
    for index, row in market_data.iterrows():
        current_price = row['hourly_avg_yes_price']
        predicted_high = row['predicted_high']
        
        # --- THE BUY SIGNAL ---
        is_target = check_bracket_match(predicted_high, ticker)
        
        # Buy if it matches our physics model AND is priced between 5c and 30c
        if not holding and is_target and 15 < current_price < 50: 
            buy_price = current_price
            holding = True
            continue 
            
        # --- THE SELL SIGNALS ---
        if holding:
            # Take-Profit: Lock in a 15-cent win instantly
            if current_price >= buy_price + 25:
                profit_cents = current_price - buy_price
                current_bankroll += (profit_cents / 100) 
                winning_trades += 1
                holding = False
                break 
                
            # Stop-Loss: Cut the cord if it drops 10 cents
            elif current_price <= buy_price - 5:
                loss_cents = current_price - buy_price # This will be a negative number
                current_bankroll += (loss_cents / 100)
                losing_trades += 1
                holding = False
                break 
                
    # --- THE TIME-STOP ---
    # Dump the contract at 11 AM if we are still holding it
    if holding:
        final_price = market_data.iloc[-1]['hourly_avg_yes_price']
        pnl_cents = final_price - buy_price
        current_bankroll += (pnl_cents / 100)
        
        if pnl_cents > 0: 
            winning_trades += 1
        else: 
            losing_trades += 1

# 3. The Final Report
print(f"\n--- Swing Trading Simulation Complete ---")
print(f"Total Trades Executed: {winning_trades + losing_trades}")
print(f"Winning Trades: {winning_trades}")
print(f"Losing Trades: {losing_trades}")
win_rate = (winning_trades / (winning_trades + losing_trades)) * 100 if (winning_trades + losing_trades) > 0 else 0
print(f"Win Rate: {win_rate:.2f}%")
print(f"Final Bankroll: ${current_bankroll:.2f}")

Initializing Kalshi Swing Trading Simulator...

--- Swing Trading Simulation Complete ---
Total Trades Executed: 48
Winning Trades: 15
Losing Trades: 33
Win Rate: 31.25%
Final Bankroll: $99.76
